In [1]:
import os

In [2]:
import os

os.chdir("/Users/basazinbelhu/Simple_MLOPS")
print(os.getcwd())

/Users/basazinbelhu/Simple_MLOPS


In [3]:
%pwd

'/Users/basazinbelhu/Simple_MLOPS'

In [4]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    ingested_train_dir: Path
    ingested_test_dir: Path

In [5]:
from src.simple_mlops.constants import *
from src.simple_mlops.utils.common import read_yaml, create_directories
# from src.simple_mlops.entity.config_entity import DataIngestionConfig

In [6]:
from src.simple_mlops.constants import *

print(CONFIG_FILE_PATH)
print(PARAMS_FILE_PATH)
print(SCHEMA_FILE_PATH)

config/config.yaml
params.yaml
schema.yaml


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifact_dir])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
            ingested_train_dir=config.ingested_train_dir,
            ingested_test_dir=config.ingested_test_dir
        )

        return data_ingestion_config

In [8]:
import urllib.request as requests

from src.simple_mlops import logger
import zipfile

In [12]:
## data ingestion component
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    #download data from source url to local data file
    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            print(f"Downloading from {self.config.source_URL} -> {self.config.local_data_file}")
            filename, headers = urllib.request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file,
            )
            logger.info(f"Downloaded {filename} with headers:\n{headers}")
        else:
            logger.info(f"File already exists at {self.config.local_data_file}")
    #unzip data from local data file to unzip dir
    def unzip_data(self):
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.unzip_data()
except Exception as e:
    logger.exception(e)

[2026-06-12 08:07:07,897: INFO: common]: Successfully read YAML file: config/config.yaml
[2026-06-12 08:07:07,899: INFO: common]: Successfully read YAML file: params.yaml
[2026-06-12 08:07:07,900: INFO: common]: Successfully read YAML file: schema.yaml
[2026-06-12 08:07:07,901: INFO: common]: Directory created or already exists: artifact
[2026-06-12 08:07:07,902: INFO: common]: Directory created or already exists: artifact/data_ingestion
[2026-06-12 08:07:09,829: INFO: 4181107048]: Downloaded data from https://archive.ics.uci.edu/static/public/186/wine+quality.zip to artifact/data_ingestion/wine_quality.zip


sh: wget: command not found
